# Task 1 – LoRA fine-tuning of Qwen2.5-0.5B-Instruct (MASSIVE → JSON)

Run on a **Colab T4 GPU** (Runtime → Change runtime type → T4 GPU), top to bottom.

Steps: check GPU → install → clone repo → load data → LoRA training → evaluate base vs fine-tuned → download results.

Settings (approved, see README for reasons): r=16, alpha=32, dropout=0.05, all linear layers, lr=2e-4 (cosine, warmup), 2 epochs, effective batch 16, loss on the JSON answer only.

In [ ]:
!nvidia-smi

In [ ]:
# Colab already has torch / transformers / datasets / accelerate. TRL and PEFT are added here.
!pip install -q -U trl peft
# Colab's preinstalled torchao is too old for the new peft and we don't use it.
!pip uninstall -y -q torchao

import torch, transformers, trl, peft, datasets
for lib in (torch, transformers, trl, peft, datasets):
    print(f"{lib.__name__:<13} {lib.__version__}")

In [ ]:
# Change this to your own repository URL.
REPO_URL = "https://github.com/<your-username>/llm-practical.git"

!git clone {REPO_URL}
%cd llm-practical

## 1. Load the processed data and turn it into prompt / completion pairs

In [ ]:
import json, sys
sys.path.insert(0, "task1_lora")
from prompt import build_messages
from datasets import load_dataset

with open("task1_lora/data/processed/labels.json") as f:
    labels = json.load(f)

raw = load_dataset("json", data_files={
    "train": "task1_lora/data/processed/train.jsonl",
    "val": "task1_lora/data/processed/val.jsonl",
})

def to_prompt_completion(row):
    # Same prompt as evaluate.py uses. The completion is the gold JSON string.
    return {
        "prompt": build_messages(row["utt"], labels["intents"], labels["slot_types"]),
        "completion": [{"role": "assistant", "content": row["target"]}],
    }

train_ds = raw["train"].map(to_prompt_completion, remove_columns=raw["train"].column_names)
val_ds = raw["val"].map(to_prompt_completion, remove_columns=raw["val"].column_names)
print(train_ds, val_ds)
print(train_ds[0]["completion"])

In [ ]:
# Check how long the prompts are, to pick max_length safely.
from transformers import AutoTokenizer
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

lengths = []
for ex in train_ds:
    text = tokenizer.apply_chat_template(ex["prompt"] + ex["completion"], tokenize=False)
    lengths.append(len(tokenizer(text)["input_ids"]))
print("tokens per example: min", min(lengths), "max", max(lengths), "mean", round(sum(lengths) / len(lengths)))

## 2. LoRA training

In [ ]:
import math, time
from peft import LoraConfig
from transformers import AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer

# fp32 master weights + fp16 mixed precision (the T4 has no bf16 support).
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

EPOCHS = 2
BATCH_SIZE = 8
GRAD_ACCUM = 2
steps_per_epoch = math.ceil(len(train_ds) / (BATCH_SIZE * GRAD_ACCUM))
total_steps = steps_per_epoch * EPOCHS
warmup_steps = max(1, round(0.03 * total_steps))  # 3% warmup
print("total optimizer steps:", total_steps, "warmup steps:", warmup_steps)

ADAPTER_DIR = "task1_lora/outputs/qwen05b-lora"
config = SFTConfig(
    output_dir=ADAPTER_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    per_device_eval_batch_size=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    max_length=512,
    completion_only_loss=True,   # loss on the JSON answer only, not the prompt
    fp16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="no",
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.model.print_trainable_parameters()

In [ ]:
torch.cuda.reset_peak_memory_stats()
start = time.time()
trainer.train()
train_minutes = (time.time() - start) / 60
peak_gb = torch.cuda.max_memory_allocated() / 1024**3
print(f"training time: {train_minutes:.1f} min, peak GPU memory allocated: {peak_gb:.2f} GB")

trainer.save_model(ADAPTER_DIR)  # saves only the LoRA adapter weights

In [ ]:
# Save the loss curve and run info for the report.
import os
os.makedirs("task1_lora/results", exist_ok=True)
run_info = {
    "train_rows": len(train_ds),
    "val_rows": len(val_ds),
    "total_steps": total_steps,
    "train_minutes": round(train_minutes, 1),
    "peak_gpu_memory_gb": round(peak_gb, 2),
    "gpu": torch.cuda.get_device_name(0),
    "versions": {lib.__name__: lib.__version__ for lib in (torch, transformers, trl, peft)},
    "log_history": trainer.state.log_history,
}
with open("task1_lora/results/train_log.json", "w") as f:
    json.dump(run_info, f, indent=2)

for entry in trainer.state.log_history:
    if "eval_loss" in entry:
        print(f"step {entry['step']:>4}  eval_loss {entry['eval_loss']:.4f}")

## 3. Evaluate base vs fine-tuned on the 200 held-out test rows

In [ ]:
# Free the training model's memory first.
del trainer, model
import gc; gc.collect(); torch.cuda.empty_cache()

# Quick smoke test on 16 rows, then the full run.
!python task1_lora/evaluate.py --adapter {ADAPTER_DIR} --limit 16

In [ ]:
!python task1_lora/evaluate.py --adapter {ADAPTER_DIR}

## 4. Download results (and the adapter) to commit locally

In [ ]:
!zip -r -q task1_results.zip task1_lora/results
!zip -r -q task1_adapter.zip {ADAPTER_DIR}
from google.colab import files
files.download("task1_results.zip")
files.download("task1_adapter.zip")